# W1.2 bulk OCR: Bodhan IndicOCR over the scanned corpus (Colab or Kaggle)

Madhya Kanoon Saathi, DEC-001 / DEC-053 / DEC-069. Needs a **GPU (T4)**. 21 documents, 635 pages, split into two packages:
`mks_ocr_bulk_p1` (19 docs, 316 pages: all indexed documents first) and `mks_ocr_bulk_p2` (2 docs, 319 pages).
Run one part per session/account; upload only that part (or set `PART` in the first cell).
The notebook detects whether it runs on Kaggle or Colab; nothing else changes. Nothing here prints secrets.

**Kaggle (DEC-069)**
1. kaggle.com -> Datasets -> New Dataset -> upload `data/ocr/bulk/mks_ocr_bulk_p1.zip` (or `_p2`) -> visibility **Private**.
2. Code -> New Notebook -> File -> Import Notebook -> this file.
3. Right panel: **Add Input** -> your dataset; Settings -> Accelerator **GPU T4 x2**; Internet **On**.
4. Add-ons -> Secrets -> add `HF_TOKEN` (Hugging Face read token) and tick it for this notebook.
5. **Save Version -> Save & Run All (Commit)**. It runs in the background and stops itself after
   `MAX_HOURS` so the output is saved. Output tab -> download `mks_ocr_bulk_pN_out.zip` when "pages missing: 0".
6. Not finished? Add the finished version's **output** as an extra input (Add Input -> Notebook Output)
   and Save & Run All again: done pages are copied over and skipped.

**Colab**: upload the unzipped `mks_ocr_bulk_p1/` (or `_p2`) to Drive root, secret `HF_TOKEN`, runtime **T4 GPU**,
Run all; re-run all after a timeout. The ZIP lands in Drive root.

Unzip each output into `data/ocr/bulk/` in the repo: `mks_ocr_bulk_pN/out/<doc>/pNNNN.md` + `.json`, `run_info.json`.

## 0. Platform and GPU check (stops early on a CPU runtime)

In [ ]:
import datetime
import hashlib
import json
import pathlib
import shutil
import time

import torch

KAGGLE = pathlib.Path("/kaggle/input").exists()
PART = None  # only if several parts are uploaded at once: 1 or 2 (mks_ocr_bulk_p1 / _p2)
MAX_HOURS = 11.0  # Kaggle stops a session at its limit (~12 h, not verified); stop before it

# Bodhan needs a GPU. A CPU runtime has a CPU-only torch and fails with
# "Torch not compiled with CUDA enabled".
assert torch.cuda.is_available(), (
    "No GPU. Kaggle: Settings -> Accelerator -> GPU T4 x2. "
    "Colab: Runtime -> Change runtime type -> T4 GPU."
)
GPU = torch.cuda.get_device_name(0)
if torch.cuda.get_device_capability(0) < (7, 0):
    print("WARNING: old GPU", GPU, "- recent torch may not support it; prefer GPU T4")
print("platform", "kaggle" if KAGGLE else "colab", "| torch", torch.__version__, "| GPU", GPU)

## 1. Inputs, outputs, resume, sha256 check

In [ ]:
def find_package(base):
    # a package = folder mks_ocr_bulk* holding jobs.json + pdfs/ (one part of the split set)
    found = sorted(
        p.parent
        for p in base.rglob("jobs.json")
        if (p.parent / "pdfs").is_dir() and p.parent.name.startswith("mks_ocr_bulk")
    )
    if PART is not None:
        found = [f for f in found if f.name.endswith(f"_p{PART}")]
    names = [str(f) for f in found]
    assert len(found) == 1, f"expected one package folder, found {names}; set PART"
    return found[0]


if KAGGLE:
    # input dataset is read-only; outputs go to /kaggle/working (saved with the version)
    IN = find_package(pathlib.Path("/kaggle/input"))
    BASE = pathlib.Path("/kaggle/working")
    OUT = BASE / IN.name / "out"
    OUT.mkdir(parents=True, exist_ok=True)
    # resume: copy pages from a previous version's output attached as input
    for prev in pathlib.Path("/kaggle/input").rglob("run_info.json"):
        if prev.parent.name == "out" and prev.parent.parent.name == IN.name:
            shutil.copytree(prev.parent, OUT, dirs_exist_ok=True)
            print("resumed from", prev.parent)
else:
    from google.colab import drive

    drive.mount("/content/drive")
    BASE = pathlib.Path("/content/drive/MyDrive")
    IN = find_package(BASE)
    OUT = IN / "out"
    OUT.mkdir(exist_ok=True)
print("package", IN.name)

JOBS = json.loads((IN / "jobs.json").read_text(encoding="utf-8"))
INFO_PATH = OUT / "run_info.json"
INFO = json.loads(INFO_PATH.read_text()) if INFO_PATH.exists() else {"pages": {}}
INFO.setdefault("sessions", []).append(
    {
        "platform": "kaggle" if KAGGLE else "colab",
        "gpu": GPU,
        "torch": torch.__version__,
        "started_at": datetime.datetime.now(datetime.UTC).isoformat(),
    }
)


def save_info():
    INFO_PATH.write_text(json.dumps(INFO, indent=1, ensure_ascii=False))


def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


bad = [j["doc_id"] for j in JOBS["jobs"] if sha256(IN / j["file"]) != j["sha256"]]
assert not bad, f"sha256 mismatch (re-upload these PDFs): {bad}"
save_info()
print(len(JOBS["jobs"]), "documents,", JOBS["total_pages"], "pages; all sha256 match")

## 2. Renderer: same pypdfium2 version as the repo lockfile, 300 dpi

In [ ]:
!pip -q install "{JOBS['renderer']}"
import pypdfium2 as pdfium

INFO["renderer"] = JOBS["renderer"]
INFO["dpi"] = JOBS["dpi"]
save_info()

## 3. Bodhan IndicOCR, pinned to the bake-off revision
Usage follows the model card at https://huggingface.co/bodhan-ai/indic-ocr (checked 2026-11-10).

In [ ]:
from huggingface_hub import snapshot_download

if KAGGLE:
    from kaggle_secrets import UserSecretsClient

    tok = UserSecretsClient().get_secret("HF_TOKEN")
else:
    from google.colab import userdata

    tok = userdata.get("HF_TOKEN")
# same revision as the W0.5 bake-off run (data/ocr/out/run_info.json)
REVISION = "cd50d301d0e17e8ecb32fc49c8ccbd7914dcfc25"
REPO = snapshot_download("bodhan-ai/indic-ocr", revision=REVISION, token=tok)
del tok
INFO["bodhan"] = {
    "repo": "bodhan-ai/indic-ocr",
    "revision": REVISION,
    "licence": "Indic Open Model License v1.0",
    "table_format": "markdown",
}
save_info()
print("revision", REVISION)

In [ ]:
!cd "$REPO" && bash install.sh

## 4. OCR every page (skips pages already done; stops after MAX_HOURS)

In [ ]:
import sys

sys.path.insert(0, REPO)
from indic_ocr import IndicOCR

parser = IndicOCR.from_pretrained(REPO, table_format="markdown")
TMP = pathlib.Path("/tmp/mks_page.png")
started = time.time()
stopped = False
for j in JOBS["jobs"]:
    d = OUT / j["doc_id"].replace("@", "_")
    d.mkdir(exist_ok=True)
    pdf = None
    for n in range(1, j["pages"] + 1):
        key = f"{j['doc_id']}#p{n}"
        if (d / f"p{n:04d}.md").exists():
            continue
        if time.time() - started > MAX_HOURS * 3600:
            stopped = True
            break
        if pdf is None:
            pdf = pdfium.PdfDocument(str(IN / j["file"]))
        img = pdf[n - 1].render(scale=JOBS["dpi"] / 72).to_pil().convert("RGB")
        img.save(TMP)
        t0 = time.time()
        page = parser.parse(str(TMP))
        js = json.dumps(page, ensure_ascii=False, indent=1)
        (d / f"p{n:04d}.json").write_text(js, encoding="utf-8")
        # the .md is written last, so its presence means the page is done
        (d / f"p{n:04d}.md").write_text(page["markdown"], encoding="utf-8")
        INFO["pages"][key] = {"seconds": round(time.time() - t0, 1), "size": list(img.size)}
        save_info()
        print(key, f"{INFO['pages'][key]['seconds']}s", len(page["markdown"]), "chars")
    if pdf is not None:
        pdf.close()
    if stopped:
        print(f"stopped after {MAX_HOURS} h; run again to continue")
        break
print("session done in", round((time.time() - started) / 60, 1), "min")

## 5. Check and package

In [ ]:
missing = [
    f"{j['doc_id']}#p{n}"
    for j in JOBS["jobs"]
    for n in range(1, j["pages"] + 1)
    if not (OUT / j["doc_id"].replace("@", "_") / f"p{n:04d}.md").exists()
]
print("pages missing:", len(missing), missing[:10])
if not missing:
    INFO["finished_at"] = datetime.datetime.now(datetime.UTC).isoformat()
    save_info()
    shutil.copy(IN / "jobs.json", OUT / "jobs.json")
    # zip holds <package>/out/..., so the two parts unzip side by side in data/ocr/bulk/
    zbase = str(BASE / f"{IN.name}_out")
    print(shutil.make_archive(zbase, "zip", OUT.parent.parent, f"{IN.name}/out"))